<a href="https://colab.research.google.com/github/Abbhijeet-018/silent-co-driver/blob/main/Copy_of_speech_emotion_recognition_with_openai_whisper_large_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("audio-classification", model="firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3")

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForAudioClassification

processor = AutoProcessor.from_pretrained("firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3")
model = AutoModelForAudioClassification.from_pretrained("firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3", device_map="auto")

In [ ]:
# Requires: librosa
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import librosa
import torch
import numpy as np

model_id = "firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3"
model = AutoModelForAudioClassification.from_pretrained(model_id)

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True)
id2label = model.config.id2label
def preprocess_audio(audio_path, feature_extractor, max_duration=30.0):
    audio_array, sampling_rate = librosa.load(audio_path, sr=None)

    max_length = int(feature_extractor.sampling_rate * max_duration)
    if len(audio_array) > max_length:
        audio_array = audio_array[:max_length]
    else:
        audio_array = np.pad(audio_array, (0, max_length - len(audio_array)))

    inputs = feature_extractor(
        audio_array,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )
    return inputs
def predict_emotion(audio_path):

    inputs = preprocess_audio(
        audio_path,
        feature_extractor
    )

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.to(device)

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )

    predicted_id = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    predicted_label = id2label[predicted_id]

    confidence = probabilities[0][predicted_id].item()

    return {
        "emotion": predicted_label,
        "confidence": confidence
    }

In [ ]:
!pip install -q fastapi uvicorn python-multipart pyngrok

In [ ]:
from fastapi import FastAPI, UploadFile, File
import tempfile
import os

app = FastAPI()


@app.get("/")
def home():
    return {
        "status": "online",
        "message": "Driver Emotion API is running"
    }


@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    # Create temporary audio file
    suffix = os.path.splitext(file.filename)[1] or ".wav"

    with tempfile.NamedTemporaryFile(
        delete=False,
        suffix=suffix
    ) as temp_file:

        temp_file.write(await file.read())
        audio_path = temp_file.name

    try:

        # YOUR EXISTING MODEL FUNCTION
        result = predict_emotion(audio_path)

        return {
            "success": True,
            "filename": file.filename,
            "emotion": result["emotion"],
            "confidence": result["confidence"]
        }

    except Exception as e:

        return {
            "success": False,
            "error": str(e)
        }

    finally:

        if os.path.exists(audio_path):
            os.remove(audio_path)

In [ ]:
import threading
import uvicorn
from pyngrok import ngrok

# Start FastAPI
def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()
ngrok.set_auth_token("3HmGYsWbY3kZM0gvxLk3JpjALiB_6scKnEEgmvBVta3WsizF3")
public_url = ngrok.connect(8000)

print("====================================")
print("Your API URL:")
print(public_url)
print("====================================")
print("Prediction endpoint:")
print(f"{public_url}/predict")